In [1]:
!pip install -q arxiv PyMuPDF language-tool-python transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 67.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 44.7 MB/s eta 0:00:00:00:0100:01


In [2]:
import arxiv

search = arxiv.Search(
    query="machine learning",
    max_results=5,
    sort_by=arxiv.SortCriterion.SubmittedDate
)

client = arxiv.Client()
papers = list(client.results(search))

for p in papers:
    print(p.title)

HTTPError: Page request resulted in HTTP 503 (https://export.arxiv.org/api/query?search_query=machine+learning&id_list=&sortBy=submittedDate&sortOrder=descending&start=0&max_results=100)

In [3]:
import time
time.sleep(30)

import arxiv

client = arxiv.Client(page_size=5, delay_seconds=5, num_retries=3)
search = arxiv.Search(
    query="machine learning",
    max_results=5,
    sort_by=arxiv.SortCriterion.SubmittedDate
)

papers = list(client.results(search))
for p in papers:
    print(p.title)

HTTPError: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=machine+learning&id_list=&sortBy=submittedDate&sortOrder=descending&start=0&max_results=5)

In [4]:
import urllib.request
import time

paper_ids = ["2301.07041", "2303.08774", "2310.06825", "2312.11805", "2401.02954"]

papers_data = []
for pid in paper_ids:
    url = f"http://export.arxiv.org/abs/{pid}"
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=15) as response:
            html = response.read().decode('utf-8', errors='ignore')
            print(f"{pid}: fetched OK, length={len(html)}")
            papers_data.append((pid, html))
    except Exception as e:
        print(f"{pid}: FAILED - {e}")
    time.sleep(3)

2301.07041: fetched OK, length=41421
2303.08774: fetched OK, length=88418
2310.06825: fetched OK, length=42379
2312.11805: fetched OK, length=266719
2401.02954: fetched OK, length=54275


In [5]:
import re

def extract_title_abstract(html):
    title_match = re.search(r'<title>(.*?)</title>', html, re.DOTALL)
    title = title_match.group(1).strip() if title_match else "Unknown"
    title = re.sub(r'^\[.*?\]\s*', '', title)  # remove "[2301.07041] " prefix

    abstract_match = re.search(r'<blockquote class="abstract mathjax">(.*?)</blockquote>', html, re.DOTALL)
    if abstract_match:
        abstract = abstract_match.group(1)
        abstract = re.sub(r'<.*?>', '', abstract)  # strip HTML tags
        abstract = re.sub(r'\s+', ' ', abstract).strip()
        abstract = abstract.replace('Abstract:', '').strip()
    else:
        abstract = "Not found"

    return title, abstract

papers = []
for pid, html in papers_data:
    title, abstract = extract_title_abstract(html)
    papers.append({'id': pid, 'title': title, 'abstract': abstract})
    print(f"{pid}: {title}")
    print(f"   Abstract length: {len(abstract)} chars")

2301.07041: Verifiable Fully Homomorphic Encryption
   Abstract length: 1417 chars
2303.08774: GPT-4 Technical Report
   Abstract length: 880 chars
2310.06825: Mistral 7B
   Abstract length: 671 chars
2312.11805: Gemini: A Family of Highly Capable Multimodal Models
   Abstract length: 1007 chars
2401.02954: DeepSeek LLM: Scaling Open-Source Language Models with Longtermism
   Abstract length: 1150 chars


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    trust_remote_code=True,
    attn_implementation="eager"
)
print("Model loaded")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

... (3 outputs left out)

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

KeyError: 'type'

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda"
)
print("Model loaded")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded


In [8]:
def score_paper(title, abstract):
    prompt = f"""You are an expert academic reviewer. Evaluate the following paper abstract on two dimensions, each scored 1-10:
1. GRAMMAR: quality of writing, sentence structure, clarity (10 = flawless, 1 = very poor)
2. REASONING: logical coherence, how well the argument/contribution is justified (10 = excellent, 1 = very poor)

Title: {title}
Abstract: {abstract}

Respond in EXACTLY this format, nothing else:
GRAMMAR: <score>
REASONING: <score>
EXPLANATION: <one sentence explaining the reasoning score>"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=True)

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

for paper in papers:
    print(f"=== {paper['title']} ===")
    result = score_paper(paper['title'], paper['abstract'])
    print(result)
    print()

=== Verifiable Fully Homomorphic Encryption ===
GRAMMAR: 8
REASONING: 7

Explanation: The abstract effectively communicates the main points about the research topic while maintaining a balanced tone without excessive jargon or complexity. It clearly outlines the motivation behind the study, describes the current state of the field, presents challenges, and proposes a solution. However, some sentences could be clearer or more concise, particularly regarding the specific details of the proposed solution and its evaluation.

=== GPT-4 Technical Report ===
GRAMMAR: 8
REASONING: 7

Explanation: The abstract provides a clear and concise summary of the research conducted using GPT-4, a large-scale, multimodal language model. It effectively communicates the capabilities and limitations of the model while highlighting its strengths in certain domains. However, there could be slight improvements in grammar and sentence structure to enhance readability and flow. Additionally, the reasoning sectio

In [9]:
!pip install language-tool-python --quiet -q

In [10]:
import language_tool_python

tool = language_tool_python.LanguageTool('en-US')

for paper in papers:
    matches = tool.check(paper['abstract'])
    num_errors = len(matches)
    print(f"{paper['title']}: {num_errors} grammar issues detected")

Verifiable Fully Homomorphic Encryption: 1 grammar issues detected
GPT-4 Technical Report: 1 grammar issues detected
Mistral 7B: 1 grammar issues detected
Gemini: A Family of Highly Capable Multimodal Models: 2 grammar issues detected
DeepSeek LLM: Scaling Open-Source Language Models with Longtermism: 0 grammar issues detected


In [11]:
from scipy.stats import pearsonr, spearmanr

llm_grammar_scores = [8, 8, 9, 9, 8]
tool_error_counts = [1, 1, 1, 2, 0]

# Note: higher LLM score should correlate with LOWER tool error count (inverse relationship expected)
pearson_r, pearson_p = pearsonr(llm_grammar_scores, tool_error_counts)
spearman_r, spearman_p = spearmanr(llm_grammar_scores, tool_error_counts)

print(f"Pearson correlation: r={pearson_r:.3f}, p={pearson_p:.3f}")
print(f"Spearman correlation: r={spearman_r:.3f}, p={spearman_p:.3f}")

Pearson correlation: r=0.645, p=0.239
Spearman correlation: r=0.645, p=0.239
